# 迭代器与生成器

学习目标：理解逐项取值与暂停执行，编写迭代器和生成器，并正确处理耗尽、双向通信及资源清理。

前置知识：容器与对象引用、for 循环、函数与作用域、类与特殊方法、异常处理和 try/finally、文本流读写。

运行环境：Python 3.12。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

## 1 可迭代对象与迭代器

逐条处理任务时，任务列表保存数据，迭代器保存当前取到哪里。iter 获取迭代器，next 从迭代器取下一项。

| 原文名称 | 中文名称／含义 |
| --- | --- |
| iterable | 可迭代对象，能够逐项提供元素的对象 |
| iterator | 迭代器，记录一次遍历的状态并提供下一项 |
| iteration | 迭代，逐项取值的过程 |

迭代器也是可迭代对象，但可迭代对象不一定是迭代器。list 既是容器，也是可迭代对象；它自身不支持 next。对列表再次调用 iter 会得到独立的迭代器，对迭代器调用 iter 则返回它自身。

In [1]:
tasks = ["阅读", "练习"]
first = iter(tasks)
second = iter(tasks)

print(next(first), next(first))  # 阅读 练习
print(next(second))  # 阅读：second 有自己的遍历位置。
print(iter(first) is first)  # True：不会把 first 重置到开头。

阅读 练习
阅读
True


In [2]:
# 预期 TypeError：直接观察本节说明的原始异常，然后继续下一单元。
next(tasks)

TypeError: 'list' object is not an iterator

## 2 next、耗尽与默认值

next(iterator, default) 中，iterator 是迭代器，default 是耗尽后返回的备用值。不提供 default 时，耗尽会引发 StopIteration；已耗尽的迭代器以后仍应报告耗尽。

默认值只处理耗尽，不会屏蔽取值过程中的其他异常。None 也可能是合法元素；需要区分“取到了 None”和“没有元素”时，可用单独创建的 object 作为标记，并用 is 比较身份。

In [3]:
items = iter([None])
missing = object()

print(next(items) is None)  # True：这是实际元素。
print(next(items, missing) is missing)  # True：现在已经耗尽。
print(next(items, "结束"))  # 结束：再次取值仍然耗尽。

True
True
结束


In [4]:
# 预期 StopIteration：直接观察本节说明的原始异常，然后继续下一单元。
next(items)

StopIteration: 

## 3 for 如何使用迭代协议

for 先对可迭代对象获取迭代器，再反复取下一项；耗尽时结束循环。循环体执行失败和正常耗尽是两回事。

下例用 iter、next 和 while 展开普通 for 的取值过程。只在 next 周围捕获 StopIteration，避免把循环体自身的问题误当作迭代结束。

In [5]:
names = ["序列", "流"]
manual = []
cursor = iter(names)

while True:
    try:
        name = next(cursor)
    except StopIteration:
        break
    manual.append(name.upper())

automatic = []
for name in names:
    automatic.append(name.upper())

print(manual)  # ['序列', '流']
print(manual == automatic)  # True：两种写法取得相同元素。

['序列', '流']
True


## 4 自定义迭代对象

### 4.1 最小迭代器

对象协议章节已经介绍通过特殊方法参与 Python 操作；这里具体实现逐项取值和耗尽状态。

| 方法／异常 | 中文名称／含义 | 约定 |
| --- | --- | --- |
| \_\_iter\_\_ | 获取迭代器的方法 | 迭代器自身应返回 self |
| \_\_next\_\_ | 获取下一项的方法 | 返回下一项，或在耗尽时引发 StopIteration |
| StopIteration | 迭代结束异常 | 一旦报告耗尽，后续取值仍应报告耗尽 |

Countdown 表示倒计时，start 是非负整数起点。它依次产生起点到 1，并把当前位置保存在实例属性中；不需要继承某个专门的迭代器基类。

In [6]:
class Countdown:
    """从非负整数起点逐次倒数到 1。"""

    def __init__(self, start):
        self.remaining = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.remaining <= 0:
            raise StopIteration
        current = self.remaining
        self.remaining -= 1
        return current


countdown = Countdown(3)
print(iter(countdown) is countdown)  # True
print(next(countdown), list(countdown))  # 3 [2, 1]
print(next(countdown, "结束"), next(countdown, "结束"))  # 结束 结束
print(list(Countdown(0)))  # []：零起点直接耗尽。

True
3 [2, 1]
结束 结束
[]


### 4.2 可重复遍历的对象

如果一个对象希望每次遍历都从头开始，其 \_\_iter\_\_ 应创建新迭代器，而不是反复返回同一个已使用的迭代器。

下面的 CountdownSeries 保存起点；每次调用 iter 时返回前文定义的 Countdown 新实例。保存数据的对象与保存遍历位置的对象由此分开。

In [7]:
class CountdownSeries:
    """保存倒计时起点，每次遍历创建独立迭代器。"""

    def __init__(self, start):
        self.start = start

    def __iter__(self):
        return Countdown(self.start)


series = CountdownSeries(2)
left = iter(series)
right = iter(series)

print(left is right)  # False
print(next(left), next(right))  # 2 2：位置互不影响。
print(list(series), list(series))  # [2, 1] [2, 1]

False
2 2
[2, 1] [2, 1]


### 4.3 序列协议的备用入口

单参数 iter 还支持没有 \_\_iter\_\_、但按序列语义实现 \_\_getitem\_\_ 的对象：从整数索引 0 开始读取，遇到 IndexError 结束。两种入口都不支持时，iter 引发 TypeError。

因此，不能只看对象有没有 \_\_iter\_\_ 就断定它能否迭代。新写可迭代类时，通常直接提供 \_\_iter\_\_ 更清楚。

In [8]:
class IndexedWords:
    """仅通过序列索引提供两个词。"""

    def __getitem__(self, index):
        return ("读取", "处理")[index]


print(list(IndexedWords()))  # ['读取', '处理']：索引 2 的 IndexError 结束迭代。

['读取', '处理']


In [9]:
# 预期 TypeError：直接观察本节说明的原始异常，然后继续下一单元。
iter(42)

TypeError: 'int' object is not iterable

## 5 双参数 iter：遇到哨兵值停止

iter(callable, sentinel) 中，callable 是可以不传实参调用的对象，sentinel 是停止标记，也叫哨兵值。每次取值都会调用 callable；返回值与 sentinel 相等时结束，哨兵值本身不交给调用者。

这与 next 的默认值不同：默认值是在耗尽后返回，哨兵值是在调用结果中被识别为停止信号。传给 iter 的应是可调用对象本身，不是先调用它得到的结果。

In [10]:
source = iter([4, 2, 0, 9])
before_zero = iter(source.__next__, 0)

print(list(before_zero))  # [4, 2]：0 被读到，但不作为元素返回。
print(next(source))  # 9：双参数 iter 没有继续读取哨兵之后的数据。
print(next(before_zero, "结束"))  # 结束：这个迭代器仍保持耗尽。

[4, 2]
9
结束


## 6 生成器函数与 yield

普通 def 的函数体出现 yield，就成为生成器函数（generator function）。调用时先得到生成器对象，首次推进时才开始运行函数体；实参表达式仍会在调用时求值，并非整个调用都延后。

yield 将值交给调用者并暂停，保留局部变量与执行位置，再次推进时从暂停处继续。沿图中的往返，定位每次 next 执行的那一段。

![生成器：调用方推动一次，函数运行到下一处暂停](image/illustration/13-01-generator-resume.svg)

图示：依据 Python 生成器执行语义自行绘制的控制权往返；最后一行表示带默认值的 next 如何向调用方呈现耗尽。

生成器对象本身就是迭代器，不必手写 \_\_iter\_\_、\_\_next\_\_；yield 后不写值则产出 None。下面先核对“已创建”与“开始”的先后，再把 2、1、耗尽对应到三次 next。

In [11]:
def generate_countdown(start):
    """从非负整数起点逐次产出倒计时，并显示执行位置。"""
    print("开始")
    remaining = start
    while remaining > 0:
        yield remaining
        remaining -= 1
    print("结束")


steps = generate_countdown(2)
print("已创建")  # 先输出此行；函数体尚未打印“开始”。
print(iter(steps) is steps)  # True：生成器对象是迭代器。
print(next(steps))  # 先打印“开始”，再得到 2。
print(next(steps))  # 1：局部变量和上次暂停的位置被保留。
print(next(steps, "耗尽"))  # 先打印“结束”，再得到“耗尽”。

已创建
True
开始
2
1
结束
耗尽


## 7 生成器如何结束

### 7.1 return 与 StopIteration.value

yield 产出供遍历的元素，return 结束生成器。return 后的值不是额外元素，而是结束时那次 StopIteration 异常的 value 属性；省略返回值时为 None。

普通 for 和 list 转换只消费产出的元素，不收集这个返回值。需要直接观察返回值时，在导致生成器结束的那次 next 周围捕获 StopIteration。

In [12]:
def report_total():
    """产出两笔数量，并在结束时返回总数量。"""
    yield 2
    yield 3
    return 5


report = report_total()
print(next(report), next(report))  # 2 3：这两个值才是遍历元素。
try:
    next(report)
except StopIteration as exc:
    print(exc.value)  # 5：来自 return。

print(list(report_total()))  # [2, 3]：不包含 return 的 5。

2 3
5
[2, 3]


### 7.2 不用 raise StopIteration 结束生成器

自定义迭代器的 \_\_next\_\_ 可以主动引发 StopIteration；生成器函数应使用 return 或自然执行到函数末尾。

在 Python 3.12 中，生成器内部直接或间接引发的 StopIteration 如果逃出函数体，会转换为 RuntimeError，并把原异常保留为原因。间接情况包括在生成器内部对空迭代器调用没有默认值的 next。

In [13]:
def broken_generator():
    """反例：错误地用 StopIteration 主动结束生成器。"""
    yield "开始"
    raise StopIteration("误用")


broken = broken_generator()
print(next(broken))  # 开始

开始


In [14]:
# 预期 RuntimeError：直接观察本节说明的原始异常，然后继续下一单元。
# 回溯先显示原始 StopIteration，再显示转换后的 RuntimeError。
next(broken)

RuntimeError: generator raised StopIteration

In [15]:
print(next(broken, "已结束"))  # 已结束：异常已经使生成器退出。

已结束


## 8 生成器表达式与惰性求值

### 8.1 简单转换和筛选

生成器表达式（generator expression）用圆括号写简单的逐项计算。下面的 value 表示输入中的一个整数；筛选偶数后再计算平方。

列表推导式直接构造结果列表；生成器表达式创建生成器，按消费需求计算结果，这称为惰性求值（lazy evaluation）。它避免预先保存全部结果，但不保证输入或每项计算都不占内存。多步骤处理用生成器函数更易读。

In [16]:
numbers = [1, 2, 3, 4]
square_list = [value * value for value in numbers if value % 2 == 0]
square_stream = (value * value for value in numbers if value % 2 == 0)

print(square_list)  # [4, 16]：构造列表时已经完成计算。
print(next(square_stream))  # 4：先取一项。
print(list(square_stream))  # [16]：只剩未消费的结果。
print(sum(value * value for value in [2, 4]))  # 20
# 生成器表达式是调用的唯一实参时，可以省略它自身的一层圆括号。

[4, 16]
4
[16]
20


### 8.2 哪些立即求值，哪些延后

生成器表达式最左侧 for 后面的可迭代表达式会立即求值。元素计算、过滤条件以及后续 for 子句中的计算，则随着生成器推进执行；不能概括成“全部都延后”。

下面用函数打印标记来观察时机。倍率 factor 是元素计算使用的外部变量，在实际计算时读取；创建表达式并不会自动冻结这些外部变量的值。

In [17]:
def provide_numbers():
    """显示输入表达式的求值时机，返回小型列表。"""
    print("取得输入")
    return [1, 2, 3]


def keep_even(value):
    """显示过滤条件执行的时机。"""
    print("筛选", value)
    return value % 2 == 0


factor = 10
scaled = (value * factor for value in provide_numbers() if keep_even(value))
print("表达式已创建")  # 此前只打印“取得输入”，没有执行筛选。
factor = 100
print(next(scaled))  # 依次打印“筛选 1”“筛选 2”，然后输出 200。
print(list(scaled))  # 打印“筛选 3”，然后输出 []。

取得输入
表达式已创建
筛选 1
筛选 2
200
筛选 3
[]


### 8.3 异常出现的时机也不同

最左侧输入不能迭代，会在创建生成器表达式时出错；元素计算中的错误则可能到消费时才出现。读取错误不能用 next 的默认值掩盖，因为默认值只对应 StopIteration。

In [18]:
# 预期 TypeError：直接观察本节说明的原始异常，然后继续下一单元。
invalid = (value for value in 42)

TypeError: 'int' object is not iterable

In [19]:
ratios = (10 / divisor for divisor in [2, 0])
print("创建成功")  # 此时尚未做除法。
print(next(ratios))  # 5.0

创建成功
5.0


In [20]:
# 预期 ZeroDivisionError：直接观察本节说明的原始异常，然后继续下一单元。
next(ratios, "耗尽")

ZeroDivisionError: division by zero

In [21]:
print(next(ratios, "已结束"))  # 已结束：未处理的异常使生成器退出。

已结束


## 9 单次消费与按需处理

给生成器增加别名不会复制遍历状态，调用 iter 也不会重新开始。要再次处理全部数据，应重新创建生成器；确实需要反复访问结果时，可以消费一次并保存成列表。

多个生成器可以逐级连接。sum 会消费到耗尽，any 则在遇到真值时停止；短路消费可能只处理输入的一部分。下面用 seen 记录实际访问过的整数，说明处理到哪里，而不进行性能测量。

In [22]:
def observe_numbers(values, seen):
    """逐项记录已访问的整数并原样产出。"""
    for value in values:
        seen.append(value)
        yield value


seen = []
observed = observe_numbers([1, 2, 3, 4], seen)
doubled = (value * 2 for value in observed)
alias = doubled

print(any(value >= 4 for value in doubled))  # True：遇到 4 后停止。
print(seen)  # [1, 2]：还没有读取后两项输入。
print(next(alias))  # 6：与 doubled 共享位置。
print(list(doubled), seen)  # [8] [1, 2, 3, 4]
print(list(alias))  # []：不能再次遍历同一个已耗尽的生成器。

True
[1, 2]
6
[8] [1, 2, 3, 4]
[]


## 10 send：向暂停处传值

received = yield outgoing 中，outgoing 是本次产出的值，received 接收下次恢复执行时送入的值。send(value) 的 value 是送入值；send 自身返回生成器下一次产出的值，若生成器结束则引发 StopIteration。

新建生成器尚未停在任何 yield，必须先用 next 或 send(None) 启动。之后 next 与 send(None) 都让暂停的 yield 表达式取得 None。不要向尚未启动的生成器发送非 None 值。

In [23]:
def receive_once():
    """先产出就绪标记，再产出收到的值，最后结束。"""
    received = yield "就绪"
    yield received


receiver = receive_once()

In [24]:
# 预期 TypeError：直接观察本节说明的原始异常，然后继续下一单元。
receiver.send(7)

TypeError: can't send non-None value to a just-started generator

In [25]:
print(receiver.send(None))  # 就绪：正确启动。
print(receiver.send(7))  # 7：send 返回下一次产出的值。
print(next(receiver, "结束"))  # 结束

another = receive_once()
print(next(another))  # 就绪
print(next(another))  # None：next 让上一个 yield 表达式取得 None。
print(next(another, "结束"))  # 结束

就绪
7
结束
就绪
None
结束


## 11 throw：在暂停处引发异常

throw(exception) 中，exception 是异常实例。它让暂停处的 yield 表现为引发该异常；生成器如果捕获异常并继续产出，throw 返回新产出的值；如果正常结束，则引发 StopIteration；未处理的异常会传播给调用者。

Python 3.12 应使用单个异常实例，例如 ValueError("重置")。分别传异常类型、值和回溯的旧式多参数签名已弃用。

下例的生成器接收整数并累计；None 表示结束，ValueError 表示重置。

In [26]:
def running_total():
    """累计 send 送入的整数，遇到 ValueError 重置，收到 None 结束。"""
    total = 0
    while True:
        try:
            # 先向调用方交出累计值，恢复时接收 send 的值或 throw 的异常。
            amount = yield total
        except ValueError:
            total = 0
            continue
        if amount is None:
            return total
        total += amount


totals = running_total()
print(next(totals))  # 0
print(totals.send(5))  # 5
print(totals.throw(ValueError("重置")))  # 0：异常在 yield 处被处理。
print(totals.send(2))  # 2

0
5
0
2


In [27]:
# 预期 KeyError：直接观察本节说明的原始异常，然后继续下一单元。
totals.throw(KeyError("未约定的指令"))

KeyError: '未约定的指令'

In [28]:
print(next(totals, "已结束"))  # 已结束：发生未处理异常后不能继续。

已结束


## 12 close 与生成器终止

### 12.1 close 触发暂停处的清理

close 会在暂停处引发 GeneratorExit，让生成器有机会执行已经进入的 try/finally。正常处理是完成清理并退出；GeneratorExit 直接继承 BaseException，不是普通业务错误。

在 Python 3.12 中，正常完成的 close 调用返回 None，不用它接收生成器的 return 值。已退出的生成器可以再次 close，不会重新执行函数体。生成器的方法也不能在它正在执行时重入调用，否则引发 ValueError。

In [29]:
def tracked_steps(events):
    """产出一个步骤，并在退出时记录清理。"""
    try:
        yield "处理中"
    finally:
        events.append("已清理")


events = []
tracked = tracked_steps(events)
print(next(tracked))  # 处理中：此时暂停在 try 内部。
print(events)  # []：暂停还不是退出。
print(tracked.close())  # None：这是 Python 3.12 的 close 返回值。
print(events)  # ['已清理']
print(tracked.close())  # None：重复关闭不重做清理。
print(events)  # ['已清理']
print(next(tracked, "已结束"))  # 已结束

处理中
[]
None
['已清理']
None
['已清理']
已结束


### 12.2 关闭过程中不能继续产出

收到 GeneratorExit 后继续 yield 会使 close 引发 RuntimeError；清理代码中的其他异常也会传播给关闭者。不要把关闭请求当作普通数据继续处理。

下面只演示这一错误边界。第一次 close 后生成器仍停在错误的 yield 处，因此在外层 finally 中再次 close，让它退出。

In [30]:
def resist_close():
    """反例：收到关闭请求后仍错误地产出一个值。"""
    try:
        yield "开始"
    except GeneratorExit:
        yield "不该继续"


resistant = resist_close()
print(next(resistant))  # 开始

开始


In [31]:
# 预期 RuntimeError：直接观察本节说明的原始异常，然后继续下一单元。
try:
    resistant.close()
finally:
    resistant.close()  # 第一次关闭仍在 yield 处暂停，第二次使其退出。

RuntimeError: generator ignored GeneratorExit

In [32]:
print(next(resistant, "已结束"))  # 已结束

已结束


## 13 yield from：委派迭代

### 13.1 传递元素并接收返回值

yield from source 中，source 是可迭代对象，其元素逐项传给外层生成器的调用者。直接 yield source 则只把 source 对象当作一个元素产出，不会自动展开它。

委派（delegation）结束时，yield from 表达式的值来自下层 StopIteration.value；下层是生成器时，就是它的 return 值。这里复用 report_total，先传出两笔数量，再把返回的总数组织成一条摘要。

In [33]:
def keep_group():
    """把整个列表作为一个元素产出。"""
    yield [2, 3]


def summarize_quantities():
    """委派两笔数量的生成，再产出总数摘要。"""
    total = yield from report_total()
    yield f"合计：{total}"


print(list(keep_group()))  # [[2, 3]]：没有展开列表。
print(list(summarize_quantities()))  # [2, 3, '合计：5']
# 前两项来自下层 yield；摘要中的 5 来自下层 return。

[[2, 3]]
[2, 3, '合计：5']


### 13.2 委派 send、throw 和 close

yield from 还会把控制操作交给当前下层迭代器；它不只是手写 for 再 yield 的缩写。

下层支持相应方法时，send 的值和 throw 的异常会传下去；关闭外层也会调用下层的 close（如果存在）。普通列表迭代器没有 send，不能要求它接收非 None 值。下例复用 running_total，因此下层能够接收整数和重置异常。

In [34]:
def delegate_total(worker, events):
    """委派给累计生成器，结束时返回其总数，并记录外层清理。"""
    try:
        return (yield from worker)
    finally:
        events.append("外层已结束")


events = []
# 第一轮让下层自然 return，第二轮用 close 提前结束整个委派链。
worker = running_total()
delegated = delegate_total(worker, events)

print(next(delegated))  # 0：来自下层。
print(delegated.send(4))  # 4：整数送进下层。
print(delegated.throw(ValueError("重置")))  # 0：异常交给下层处理。
print(delegated.send(3))  # 3
try:
    delegated.send(None)
except StopIteration as exc:
    print(exc.value)  # 3：下层 return 经 yield from 传给外层 return。
print(events)  # ['外层已结束']

events = []
worker = running_total()
delegated = delegate_total(worker, events)
print(next(delegated))  # 0
delegated.close()
print(next(worker, "下层已结束"))  # 下层已结束：close 已向下委派。
print(events)  # ['外层已结束']

0
4
0
3
3
['外层已结束']
0
下层已结束
['外层已结束']


## 14 资源清理与提前退出

### 14.1 break 只退出循环

for 中的 break 不保证关闭所遍历的生成器。若仍保留生成器引用，它可以继续暂停，内部资源也可能仍处于打开状态；不能依赖垃圾回收的时机安排必要清理。

下面在生成器内部创建 StringIO 内存文本流。events 记录打开与关闭事件；finally 负责关闭流。流是迭代器，可以逐行读取，这里用 next 观察读取与暂停。

In [35]:
import io


def read_lines(text, events):
    """创建内存文本流，逐行产出，并在退出时关闭流。"""
    stream = io.StringIO(text)
    events.append("已打开")
    # yield 暂停时流仍然打开；finally 负责生成器真正结束时的关闭。
    try:
        for line in stream:
            yield line.strip()
    finally:
        stream.close()
        events.append(("已关闭", stream.closed))


events = []
lines = read_lines("甲\n乙\n", events)
try:
    for line in lines:
        print(line)  # 甲
        break
    print(events)  # ['已打开']：break 后没有进入生成器的 finally。
    print(next(lines))  # 乙：同一生成器仍可以继续。
finally:
    lines.close()  # 消费者明确结束使用，而不是等待垃圾回收。

print(events)  # ['已打开', ('已关闭', True)]

甲
['已打开']
乙
['已打开', ('已关闭', True)]


### 14.2 消费者也用 try/finally 明确关闭

生成器内部的 finally 定义“退出时怎么清理”，消费者的 finally 定义“什么时候结束使用”。两者配合后，正常耗尽、提前 break 和消费代码抛出异常，都有明确的关闭路径。

下面继续使用 read_lines。三个分支使用相同的小型输入，events 分别记录每次独立运行的清理情况；不需要装饰器或自定义上下文管理器。

In [36]:
for mode in ["读完", "提前退出", "消费失败"]:
    events = []
    lines = read_lines("甲\n乙\n", events)
    consumed = []
    try:
        try:
            for line in lines:
                consumed.append(line)
                if mode == "提前退出":
                    break
                if mode == "消费失败":
                    raise ValueError("处理当前行失败")
        finally:
            lines.close()
    except ValueError as exc:
        print(type(exc).__name__)  # 仅“消费失败”分支打印 ValueError。
    print(mode, consumed, events)

# 读完分支消费 ['甲', '乙']，另两条分支只消费 ['甲']。
# 三条分支均记录 ['已打开', ('已关闭', True)]，各关闭一次。

读完 ['甲', '乙'] ['已打开', ('已关闭', True)]
提前退出 ['甲'] ['已打开', ('已关闭', True)]
ValueError
消费失败 ['甲'] ['已打开', ('已关闭', True)]


### 14.3 尚未启动时，函数体也不会负责清理

关闭尚未启动的生成器不会进入它的函数体，因而不能指望其中尚未进入的 finally 释放已经在外部创建的资源。

read_lines 把流的创建放在生成器函数体内，所以尚未启动时没有获得资源，也不需要释放它。若资源由调用者预先创建，调用者应自行管理它的关闭，不能只把责任交给一个可能从未启动的生成器。

In [37]:
events = []
unused = read_lines("尚未读取\n", events)
unused.close()

print(events)  # []：既没有创建流，也没有执行内部 finally。
print(next(unused, "已结束"))  # 已结束：close 不会替它执行一次正文。

[]
已结束


## 本章小结

- 可迭代对象提供迭代入口；迭代器保存一次遍历的状态，耗尽后不能靠 iter 重置。
- 生成器对象是迭代器。yield 暂停并产出，return 结束；返回值通过结束时的 StopIteration.value 传递。
- 生成器表达式最左侧输入立即求值，逐项计算与筛选延后；惰性处理不等于冻结外部变量或自动缓存结果。
- send 向暂停处送值，throw 在暂停处引发异常，close 请求结束；yield from 可以委派元素和控制操作。
- break 不保证生成器关闭。把资源获取、内部 finally 和消费者显式 close 的责任安排清楚。

自查：能否分别指出一个生成器何时创建、何时开始执行、何时耗尽，以及提前停止后由谁关闭？

## 练习

### 练习 1：预测消费位置与变量取值

先写下三行输出及理由，再运行核对。说明 alias 是否拥有独立的遍历位置，倍率在什么时候被读取，以及最后一次 list 是否重新开始计算。

In [38]:
# 先预测三次消费的值和剩余项数，再核对共享迭代器与 factor 的读取时机。
factor = 2
values = (value * factor for value in [1, 2, 3])
alias = values
print(next(values))
factor = 10
print(list(alias))
print(list(values))

2
[20, 30]
[]


### 练习 2：实现跳步迭代器

实现 StepIterator(start, stop, step)：三个参数均为整数，start 是起点，stop 是不包含的上界，step 是正整数步长。按给定合法参数直接实现迭代协议，不使用 yield。

完成后检查：参数为 (1, 7, 2) 时依次得到 1、3、5；start 等于或大于 stop 时为空；耗尽后连续两次 next 配合默认值都返回默认值；把 step 改为 1 与 3，分别预测并核对区间 (1, 7) 内的结果。

In [39]:
class StepIterator:
    """练习占位：实现正步长、上界不包含的整数迭代器。"""

    pass


# 添加初始化、迭代协议及题目指定的边界检查。
# 当前只定义占位类，不调用尚未完成的方法。

### 练习 3：委派、返回值与提前关闭

编写 counted_lines(text, events)：调用本章的 read_lines 取得内部生成器，逐条 yield 它的结果，返回实际产出的行数，并在自己的 finally 中关闭内部生成器。再编写 outer_lines，用 yield from 委派给它并返回取得的行数。

用 "甲\n乙\n" 完整消费 outer_lines，检查两个产出值及结束时 StopIteration.value 为 2，events 显示流已关闭。再创建新的一组，只取第一行后显式 close 外层，检查资源也已关闭。

提示：计数在每次 yield 前增加；完整消费的返回值要通过最后一次 next 捕获，提前 close 不用于取得计数。

In [40]:
# 在此实现 counted_lines 和 outer_lines，并添加完整消费与提前关闭检查。
# 复用本章的 read_lines；代码完成前使用 pass 保持全篇可顺序执行。
pass

### 提示

第一题只存在一个生成器对象，逐次标记已消费元素。第二题将当前位置保存在实例中，只在到达上界时停止。第三题让每层生成器负责关闭自己持有的下层对象。

### 参考解析

第一题输出 2、[20, 30]、[]。第一次读取使用 factor=2，后续两项使用更新后的 10；alias 和 values 指向同一个生成器，最后没有剩余项。

第二题初始化保存 start、stop、step，迭代入口返回 self；取下一项时，当前位置达到 stop 就引发 StopIteration，否则保存当前值、增加 step 并返回保存的值。原参数得到 [1, 3, 5]；步长 1 得到 [1, 2, 3, 4, 5, 6]，步长 3 得到 [1, 4]。起点已达上界时自然为空，耗尽后位置不会退回。

第三题 counted_lines 在每次 yield 前增加计数，循环正常结束后 return 计数，并用 finally 关闭内部 read_lines。outer_lines 返回 yield from 的结果。完整消费得到“甲”“乙”，最后一次 next 的 StopIteration.value 为 2；只取第一行再 close 时，不读取返回值，但关闭沿委派链执行，events 同样记录 ['已打开', ('已关闭', True)]。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| Python 官方文档（docs.python.org） | Python 3.12：[可迭代对象](https://docs.python.org/3.12/glossary.html#term-iterable)、[迭代器与单次遍历](https://docs.python.org/3.12/glossary.html#term-iterator)、[生成器术语](https://docs.python.org/3.12/glossary.html#term-generator)；[iter 的单参数与双参数形式](https://docs.python.org/3.12/library/functions.html#iter)、[next 与默认值](https://docs.python.org/3.12/library/functions.html#next)、[迭代协议及持续耗尽](https://docs.python.org/3.12/library/stdtypes.html#iterator-types)、[生成器类型](https://docs.python.org/3.12/library/stdtypes.html#generator-types)；[for 的取值过程](https://docs.python.org/3.12/reference/compound_stmts.html#the-for-statement)、[break 的作用](https://docs.python.org/3.12/reference/simple_stmts.html#the-break-statement)、[finally 的清理行为](https://docs.python.org/3.12/reference/compound_stmts.html#finally-clause)；[调用前的实参求值](https://docs.python.org/3.12/reference/expressions.html#calls)、[生成器表达式的求值时机](https://docs.python.org/3.12/reference/expressions.html#generator-expressions)、[yield、暂停状态与 yield from 委派](https://docs.python.org/3.12/reference/expressions.html#yield-expressions)、[生成器方法及重入限制](https://docs.python.org/3.12/reference/expressions.html#generator-iterator-methods)、[send](https://docs.python.org/3.12/reference/expressions.html#generator.send)、[throw 与 3.12 弃用说明](https://docs.python.org/3.12/reference/expressions.html#generator.throw)、[close](https://docs.python.org/3.12/reference/expressions.html#generator.close)；[return 在生成器中的含义](https://docs.python.org/3.12/reference/simple_stmts.html#the-return-statement)、[StopIteration.value 与 RuntimeError 转换](https://docs.python.org/3.12/library/exceptions.html#StopIteration)、[GeneratorExit](https://docs.python.org/3.12/library/exceptions.html#GeneratorExit)；[any 的短路行为](https://docs.python.org/3.12/library/functions.html#any)、[sum 消费输入](https://docs.python.org/3.12/library/functions.html#sum)；[StringIO 内存文本流](https://docs.python.org/3.12/library/io.html#io.StringIO)、[流的迭代协议](https://docs.python.org/3.12/library/io.html#io.IOBase)、[流关闭](https://docs.python.org/3.12/library/io.html#io.IOBase.close)、[closed 属性](https://docs.python.org/3.12/library/io.html#io.IOBase.closed)。 |
| GitHub（github.com） | [CPython v3.12.14：Objects/genobject.c 第 374–428 行的 gen_close](https://github.com/python/cpython/blob/v3.12.14/Objects/genobject.c#L374-L428)，补充定位本章运行版本中尚未启动时直接结束、关闭下层迭代器，以及正常 close 返回 None 的实现。 |